In [7]:
import numpy as np
from numpy.typing import NDArray
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
import lightgbm as lgb

import os
import sys
root_path="/home/iaw/DATA2/AAReact/src"
sys.path.append(root_path)
from util.RegressMetrics import r2_score, mse_score, mae_score, rmse_score
from util.train_tools import build_model, search_parms, split_data, load_data, eval_dataset_split
from tool.analysis import draw_pred_result
from util.featurizer import xtb_label_search
import seaborn as sns
from matplotlib import pyplot as plt

from typing import List, Tuple
from rich.table import Table
from rich import print as rp
from rich.progress import track
import shap
import pickle

from joblib import dump, load

In [8]:
seed = 1
test_size = 0.2
prefix = "/home/iaw/DATA2/AAReact/DataSet/Data_All/4_data_for_train/rdkit_xtb/train_test"
data_x, data_y, x_label, data_class, data_name = load_data(
    "{}_data_x.npy".format(prefix), 
    "{}_data_y.npy".format(prefix), 
    "{}_x_label.pkl".format(prefix), 
    "{}_data_class.pkl".format(prefix), 
    "{}_data_name.pkl".format(prefix)
)


In [9]:
## 超参之后的参数
#xgb_best_params = {
#    "colsample_bytree": 0.9,
#    "learning_rate": 0.1,
#    "max_depth": 8,
#    "min_child_weight": 5,
#    "n_estimators": 50,
#    "reg_alpha": 0.5,
#    "reg_lambda": 1.5,
#    "subsample": 0.8
#}
#
#xgb_eval_dataset_split_result = eval_dataset_split(
#                     seed_s = [1, 2, 3, 4]
#                   , test_size_s = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
#                   , parms = xgb_best_params
#                   , model_name = "xgb"
#                   , data_x = data_x, data_y = data_y, data_class = data_class
#                   , eval_func = r2_score, n_cpu = -1) 


In [10]:
#def cos_sim(v1: NDArray, v2: NDArray) -> float:
#    """
#    计算两个一维 numpy 向量的余弦相似度
#    """
#    dot_product = np.dot(v1, v2)
#    norm1 = np.linalg.norm(v1)
#    norm2 = np.linalg.norm(v2)
#    
#    if norm1 == 0 or norm2 == 0:
#        return 0.0
#    
#    return dot_product / (norm1 * norm2)


In [11]:
#feat_importance_s = xgb_eval_dataset_split_result[-1]
#test_size_s = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
#
#ref_feat_importance = feat_importance_s["1"][1]         # seed 1, test_size 0.2
#for i_seed in feat_importance_s.keys():
#    for j in range(len(test_size_s)):
#        print("seed[{}], cos_smi[{}]-[{}]: {:.4f}".format(i_seed, j, j+1, cos_sim(ref_feat_importance, feat_importance_s[i_seed][j])))

In [12]:

#from sklearn.metrics.pairwise import cosine_similarity
#
## 工具函数：计算余弦相似度
#def cos_sim(a, b):
#    return cosine_similarity([a], [b])[0][0]
#
## ====================== 你原来的数据 ======================
#feat_importance_s = xgb_eval_dataset_split_result[-1]
#test_size_s = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
#
## 基准：seed 1, test_size 0.2
#ref_feat_importance = feat_importance_s["1"][1]
#
## ====================== 计算所有 seed 的特征重要性余弦相似度 ======================
#seed_cos_sim = {}
#for i_seed in feat_importance_s.keys():
#    sim_list = []
#    for j in range(len(test_size_s)):
#        sim = cos_sim(ref_feat_importance, feat_importance_s[i_seed][j])
#        sim_list.append(sim)
#        print(f"seed[{i_seed}], test_size_idx[{j}]: cos_sim = {sim:.4f}")
#    seed_cos_sim[i_seed] = sim_list
#
## ====================== 全局 SCI 样式（完全不变）======================
#plt.rcParams.update({
#    'font.family': 'Times New Roman',
#    'font.size': 11,
#    'axes.labelsize': 13,
#    'axes.titlesize': 13,
#    'xtick.labelsize': 10,
#    'ytick.labelsize': 10,
#    'legend.fontsize': 10,
#    'axes.linewidth': 1.2,
#    'xtick.major.width': 1.2,
#    'ytick.major.width': 1.2,
#    'figure.dpi': 300,
#    'savefig.dpi': 600,
#    'savefig.bbox': 'tight',
#    'savefig.format': 'pdf'
#})
#
## ====================== 绘图：性能曲线 + 特征重要性曲线 ======================
#plt.figure(figsize=(8, 5), dpi=300)
#
## --- 1. 训练集 R² 曲线 ---
#plt.errorbar(
#    test_size_s, xgb_eval_dataset_split_result[0], xgb_eval_dataset_split_result[1],
#    label="Train $R^2$", fmt="-o", capsize=5, capthick=1.5, elinewidth=1.5,
#    linewidth=2.5, markersize=7, markerfacecolor="#4C72B0",
#    markeredgecolor="white", markeredgewidth=1.2, color="#4C72B0"
#)
#
## --- 2. 测试集 R² 曲线 ---
#plt.errorbar(
#    test_size_s, xgb_eval_dataset_split_result[2], xgb_eval_dataset_split_result[3],
#    label="Test $R^2$", fmt="-s", capsize=5, capthick=1.5, elinewidth=1.5,
#    linewidth=2.5, markersize=7, markerfacecolor="#DD8452",
#    markeredgecolor="white", markeredgewidth=1.2, color="#DD8452"
#)
#
## --- 3. 不同 seed 的特征重要性余弦相似度曲线（修复版！）---
#colors = ['#55A868', '#C44E52', '#8172B2', '#CCB974']  # 绿、红、紫、黄
#markers = ['^', 'D', 'p', '*']
#for idx, (seed, sim_vals) in enumerate(seed_cos_sim.items()):
#    plt.plot(
#        test_size_s, sim_vals,
#        label=f"Seed {seed} Feat Importance",
#        linestyle='-',        # 去掉 fmt，分开写
#        marker=markers[idx],
#        linewidth=2.5,
#        markersize=7,
#        markerfacecolor=colors[idx],
#        markeredgecolor="white",
#        markeredgewidth=1.2,
#        color=colors[idx]
#    )
#
## --- 基准虚线 ---
#plt.plot([0, 1], [0.5, 0.5], linestyle='--', c="#000000", linewidth=1.2, label='Baseline ($R^2=0.5$)')
#
## --- 图表设置 ---
#plt.xlabel("Test Size", fontweight='bold')
#plt.ylabel("$R^2$ / Cosine Similarity", fontweight='bold')
#plt.xlim(0, 1)
#plt.ylim(0, 1)
#plt.grid(True, axis='y', linestyle='--', alpha=0.3, linewidth=0.8)
#plt.legend(frameon=True, edgecolor='black', fancybox=False, ncol=2)
#
#plt.tight_layout()
#plt.savefig('model_sensitivity_test_size.png', dpi=600)
#plt.close()

In [13]:
model = load("/home/iaw/DATA2/AAReact/train/output/pt/xgb_rdkit_xtb_seed_0-1_test_0-2.pkl")
X_train, X_test, y_train, y_test,  class_train, class_test, name_train, name_test = train_test_split(
    data_x,        
    data_y,
    data_class,
    data_name,
    test_size=test_size,
    random_state=seed, 
)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [14]:
pred_error_sign = list(np.where(train_pred*y_train < 0)[0])
print("The num of sign-error is {}".format(len(pred_error_sign)))

The num of sign-error is 21


In [ ]:

## ====================== 全局 SCI 样式（完全统一）======================
#plt.rcParams.update({
#    'font.family': 'Times New Roman',
#    'font.size': 14,
#    'axes.labelsize': 16,
#    'axes.titlesize': 16,
#    'xtick.labelsize': 13,
#    'ytick.labelsize': 13,
#    'legend.fontsize': 13,
#    'axes.linewidth': 1.2,
#    'xtick.major.width': 1.2,
#    'ytick.major.width': 1.2,
#    'figure.dpi': 300,
#    'savefig.dpi': 600,
#    'savefig.bbox': 'tight',
#    'savefig.format': 'pdf'
#})
#
## ====================== 你的数据（直接替换）======================
#y_true_s = (y_train, y_test)
#y_pred_s = (train_pred, test_pred)
#
#r2_s = (
#    r2_score(y_pred_s[0], y_true_s[0]),
#    r2_score(y_pred_s[1], y_true_s[1])
#)
#print(f"Train R²: {r2_s[0]:.4f}, Test R²: {r2_s[1]:.4f}")
#
## ====================== 绘图 ======================
#fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
#
## 配色
#color_train = "#5b3284"
#color_test  = "#982635"
#color_diag  = "#828282"
#
## 绘制散点
#ax.scatter(y_true_s[0], y_pred_s[0], 
#           c=color_train, alpha=0.9, s=40, label=f"Train ($R^2$ = {r2_s[0]:.4f})",
#           edgecolors='white', linewidth=0.5)
#ax.scatter(y_true_s[1], y_pred_s[1], 
#           c=color_test, alpha=0.9, s=40, label=f"Test ($R^2$ = {r2_s[1]:.4f})",
#           edgecolors='white', linewidth=0.5)
#
## 范围
#lims = [-1, 1]
#
## 对角线 y=x
#x_line = np.linspace(lims[0], lims[1], 100)
#ax.plot(x_line, x_line, '--', color=color_diag, linewidth=1)
#ax.plot([0, 0], [-1, 1], '-', color="black", linewidth=1.5)
#ax.plot([-1, 1], [0,0], '-', color="black", linewidth=1.5)
#
## ====================== ✅ 你要的区域：左上 + 右下 红色填充 ======================
## 左上：x from -1 to 0, y from 0 to 1
##ax.axvspan(xmin=-1, xmax=0, ymin=0.5, ymax=1, color='#f47f7e', alpha=0.1, zorder=0)
#
## 右下：x from 0 to 1, y from -1 to 0
##ax.axvspan(xmin=0, xmax=1, ymin=0, ymax=0.5, color='#f47f7e', alpha=0.1, zorder=0)
#
## ====================== 样式 ======================
#ax.set_xlabel('True Value (ee.)', fontweight='bold')
#ax.set_ylabel('Predicted Value (ee.)', fontweight='bold')
#ax.set_xlim(lims)
#ax.set_ylim(lims)
#ax.set_aspect('equal', adjustable='box')
#
##ax.grid(True, linestyle='--', alpha=0.3, lw=0.8)
#ax.legend(frameon=False, fancybox=False, loc='upper left')
#
#plt.tight_layout()
#plt.savefig('regression_parity_plot.png', dpi=600)
#plt.close()

Train R²: 0.9163, Test R²: 0.8387


In [ ]:
y_true_s = (y_train, y_test)
y_pred_s = (train_pred, test_pred)
r2_s = (
    r2_score(y_pred_s[0], y_true_s[0]),
    r2_score(y_pred_s[1], y_true_s[1])
)

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 14,
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))


color_train = "#5b3284"
color_test  = "#982635"
color_diag  = "#828282"
lims = [-1, 1]

ax.scatter(y_true_s[0], y_pred_s[0], 
           c=color_train, alpha=0.9, s=40, label=f"Train ($R^2$ = {r2_s[0]:.4f})",
           edgecolors='white', linewidth=0.5)
ax.scatter(y_true_s[1], y_pred_s[1], 
           c=color_test, alpha=0.9, s=40, label=f"Test ($R^2$ = {r2_s[1]:.4f})",
           edgecolors='white', linewidth=0.5)



x_line = np.linspace(lims[0], lims[1], 100)
ax.plot(x_line, x_line, '--', color=color_diag, linewidth=1)

ax.set_xticks([-1, -0.75, -0.5, -0.25, 0.25, 0.50, 0.75, 1.00])
ax.set_yticks([-1, -0.75, -0.5, -0.25, 0.25, 0.50, 0.75, 1.00])
ax.set_aspect('equal', adjustable='box')

ax.legend(frameon=False, fancybox=False, loc='upper left')

plt.tight_layout()
plt.savefig('regression_parity_plot.png', dpi=600)
plt.close()

In [22]:
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 14,
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

fig, ax1 = plt.subplots(figsize=(10, 8), dpi=300)

xgb_explainer = shap.TreeExplainer(model)
train_shap_values = xgb_explainer.shap_values(X_train)
labels = x_label["label1_rdkit"] + x_label["label2_xtb"]
# 绘制蜂群图
shap.summary_plot(
    train_shap_values, 
    X_train, 
    feature_names=labels, 
    plot_type="dot", 
    show=False,
    plot_size=None 
)

# 绘制条形图
ax2 = ax1.twiny()
shap.summary_plot(
    train_shap_values, 
    X_train, 
    feature_names=labels, 
    plot_type="bar", 
    show=False,
    plot_size=None
)

# 让 bar 半透明，不遮挡蜂群图
for patch in ax2.patches:
    patch.set_alpha(0.2)

# 分界线
ax2.axhline(y=20, color='black', linestyle='-', linewidth=1.0)

# 轴标签（统一论文风格）
ax1.set_xlabel('SHAP Value (Bee Swarm Plot)', fontweight='bold', labelpad=10)
ax2.set_xlabel('Feature Importance', fontweight='bold', labelpad=10)


# 上轴显示在顶部
ax2.xaxis.set_label_position('top')
ax2.xaxis.tick_top()

# 布局与保存
plt.tight_layout()
plt.savefig('shap_summary_beeswarm_bar.png', dpi=600)
plt.close()